# 05 — Boundary Probe: Where Does the Equity Signal Break Down?

**Purpose:** Round 2 (p=2000/5000/8000) kept a real equity signal; Round 3 (p=15000/20000/25000)
collapsed completely (every LSOA self-serving, M2=100%, A and D identical). This notebook only
asks *where between 8,000 and 15,000 the collapse happens* — not a headline result.

**Two deliberate shortcuts, both fine for a boundary probe:**
1. **Only Scenario A and D** (the two extremes) — if they're still different, equity survives;
   if they're identical, it's gone. B/C don't add information for this specific question.
2. **time_limit cut to 60s, frac_gap relaxed to 5%.** Round 3 showed actual wall time runs
   ~60s over whatever time_limit is set, regardless of p — that's fixed LP-construction/
   subprocess overhead (~4,994 LSOAs x 40 neighbours ≈ 200k yᵢⱼ variables), not solve time.
   We don't need a provably optimal answer here, just whether M2/slack/xⱼ-vs-income already
   look degenerate at a given p — a rough answer arrives much faster than a precise one.

K0 anchor (p_primary=250, fixed), k=40, Uj=150 — unchanged from Round 2, for comparability.

## 0. Setup and reload cleaned data

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
from scipy.stats import pearsonr
import pulp
import os
import time

os.environ.setdefault("SHAPE_RESTORE_SHX", "YES")

BASE_CANDIDATES = [
    "/Users/alexia/Documents/CASA/Dissertation",
    os.path.abspath(os.path.join(os.getcwd(), "..")),
]
BASE = next(
    (b for b in BASE_CANDIDATES
     if os.path.exists(os.path.join(b, "05_processed/demand_london.csv"))),
    BASE_CANDIDATES[0],
)
print("Using BASE:", BASE)

demand_london = pd.read_csv(os.path.join(BASE, "05_processed/demand_london.csv"))
seff_london   = pd.read_csv(os.path.join(BASE, "05_processed/seff_london.csv"))
imd_london    = pd.read_csv(os.path.join(BASE, "05_processed/imd_london_clean.csv"))
census_london = pd.read_csv(os.path.join(BASE, "05_processed/census_london_clean.csv"))
print("Datasets reloaded.")


## 1. LSOA centroids (I = J)

In [ ]:
lsoa_boundaries = gpd.read_file(os.path.join(BASE, "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp"))
if lsoa_boundaries.crs is None:
    lsoa_boundaries = lsoa_boundaries.set_crs(epsg=27700)
elif lsoa_boundaries.crs.to_epsg() != 27700:
    lsoa_boundaries = lsoa_boundaries.to_crs(epsg=27700)

london_codes = set(demand_london["lsoa_code"])
lsoa_london = lsoa_boundaries[lsoa_boundaries["LSOA21CD"].isin(london_codes)].copy()
lsoa_london = lsoa_london.rename(columns={"LSOA21CD": "lsoa_code"})[["lsoa_code", "geometry"]]
lsoa_london["centroid"] = lsoa_london.geometry.centroid
lsoa_london["cx"] = lsoa_london["centroid"].x
lsoa_london["cy"] = lsoa_london["centroid"].y

lsoa_master = lsoa_london[["lsoa_code", "cx", "cy"]].merge(
    demand_london[["lsoa_code", "D_A", "D_B", "D_C", "D_D"]], on="lsoa_code", how="inner"
).merge(seff_london[["lsoa_code", "ej"]], on="lsoa_code", how="left").reset_index(drop=True)
lsoa_master["ej"] = lsoa_master["ej"].fillna(0)
lsoa_master = lsoa_master.merge(census_london[["lsoa_code", "Hi", "Ci"]], on="lsoa_code", how="left")
lsoa_master["Vi"] = lsoa_master["Hi"] * lsoa_master["Ci"]
lsoa_master = lsoa_master.merge(imd_london[["lsoa_code", "income_decile"]], on="lsoa_code", how="left")

n = len(lsoa_master)
coords = lsoa_master[["cx", "cy"]].to_numpy()
print(f"LSOA master table: {n} LSOAs")


## 2. K0 (unchanged anchor: p_primary=250, fixed)

In [ ]:
p_primary = 250
sigma_Di = lsoa_master["D_A"].sum()
sigma_ej = lsoa_master["ej"].sum()
K0 = sigma_Di / (sigma_ej + p_primary)
print(f"K0 = {K0:.4f}")


## 3. Core functions (unchanged from Round 2/3, time_limit/frac_gap only changed at call time)

In [ ]:
def evaluate_allocation_fast(demand_col, xj, K, lsoa_master, coords):
    """Lightweight version for the boundary probe: just M1, M2, and the xj-income
    correlation ingredients. Skips M3/M4/per-decile breakdown -- not needed to answer
    'has it degenerated yet'."""
    Di = lsoa_master[demand_col].to_numpy()
    Vi = lsoa_master["Vi"].to_numpy()
    ej = lsoa_master["ej"].to_numpy()
    n = len(lsoa_master)

    capacity = ej + xj
    has_capacity = capacity > 0
    cap_positions = np.where(has_capacity)[0]
    tree_cap = cKDTree(coords[has_capacity])
    dist, nearest_pos = tree_cap.query(coords, k=1)

    M1 = float((Vi * dist).sum() / Vi.sum())
    within_800 = (dist < 800).astype(float)
    M2 = float((Vi * within_800).sum() / Vi.sum())
    n_self_served = int((dist < 1.0).sum())  # LSOAs essentially serving themselves

    return {"M1_avg_dist_m": M1, "M2_coverage_800m": M2, "n_self_served": n_self_served}


def solve_joint_p_median(demand_col, p, K, lsoa_master, coords,
                         Uj=150, k=40, feas_margin=0.02, allow_K_bump=True,
                         time_limit=60, frac_gap=0.05, msg=False):
    """Same formulation as Round 2/3. time_limit/frac_gap defaults loosened for this
    boundary probe -- see intro note on why that's an acceptable trade here."""
    Di = lsoa_master[demand_col].to_numpy(dtype=float)
    ej = lsoa_master["ej"].to_numpy(dtype=float)
    n = len(lsoa_master)
    sum_Di, sum_ej = Di.sum(), ej.sum()

    K_used = K
    if allow_K_bump:
        K_min = sum_Di / (sum_ej + p)
        K_used = max(K, K_min * (1 + feas_margin))

    tree = cKDTree(coords)
    _, nbr = tree.query(coords, k=min(k, n))
    cand = [set(np.atleast_1d(row).tolist()) for row in nbr]
    for i in range(n):
        cand[i].add(i)

    prob = pulp.LpProblem("joint_p_median", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", range(n), lowBound=0, upBound=Uj, cat="Integer")
    y = {(i, j): pulp.LpVariable(f"y_{i}_{j}", lowBound=0, upBound=1)
         for i in range(n) for j in cand[i]}
    s = pulp.LpVariable.dicts("s", range(n), lowBound=0)

    def d(i, j):
        return float(np.hypot(coords[i, 0] - coords[j, 0], coords[i, 1] - coords[j, 1]))

    max_dist = float(np.hypot(np.ptp(coords[:, 0]), np.ptp(coords[:, 1])))
    M = 100.0 * max_dist

    prob += (pulp.lpSum(Di[i] * d(i, j) * y[(i, j)] for (i, j) in y)
             + M * pulp.lpSum(s[j] for j in range(n)))

    for i in range(n):
        prob += pulp.lpSum(y[(i, j)] for j in cand[i]) == 1

    served_by = {j: [] for j in range(n)}
    for (i, j) in y:
        served_by[j].append(i)
    for j in range(n):
        if served_by[j]:
            prob += (pulp.lpSum(Di[i] * y[(i, j)] for i in served_by[j])
                     <= K_used * (ej[j] + x[j]) + s[j])

    prob += pulp.lpSum(x[j] for j in range(n)) == p

    status = prob.solve(pulp.PULP_CBC_CMD(msg=int(msg), timeLimit=time_limit, gapRel=frac_gap))
    xj = np.array([int(round(x[j].value() or 0)) for j in range(n)])
    sj = np.array([float(s[j].value() or 0) for j in range(n)])
    slack_total = float(sj.sum())
    return {
        "xj": xj, "sj": sj, "K_used": K_used,
        "status": pulp.LpStatus[status],
        "slack_total": slack_total,
        "slack_frac": slack_total / sum_Di if sum_Di else 0.0,
    }


## 4. Boundary probe: p = 9000, 11000, 13000 — Scenario A vs D only

Round 2's p=8000 was still meaningfully differentiated; Round 3's p=15000 was fully
degenerate. This brackets the middle.

In [ ]:
P_VALUES = [9000, 11000, 13000]
ALPHA_LABELS = {"A": "D_A", "D": "D_D"}   # extremes only -- see intro

boundary_results = {}
boundary_rows = []

for p in P_VALUES:
    for alpha_label, demand_col in ALPHA_LABELS.items():
        t0 = time.time()
        sol = solve_joint_p_median(demand_col, p=p, K=K0, lsoa_master=lsoa_master, coords=coords)
        xj = sol["xj"]
        ev = evaluate_allocation_fast(demand_col, xj, sol["K_used"], lsoa_master, coords)
        boundary_results[(alpha_label, p)] = {**sol, **ev, "xj": xj}
        elapsed = time.time() - t0
        print(f"p={p}, Scenario {alpha_label}: status={sol['status']}, slack={sol['slack_frac']:.2%}, "
              f"M1={ev['M1_avg_dist_m']:.1f}m, M2={ev['M2_coverage_800m']:.1%}, "
              f"self-served={ev['n_self_served']}/{len(lsoa_master)}, time={elapsed:.0f}s")

print()
print("=== Verdict per p: is A still different from D? ===")
imd_lookup = imd_london.set_index("lsoa_code")["income_score"]
lsoa_codes = lsoa_master["lsoa_code"].values
income_arr = imd_lookup.loc[lsoa_codes].to_numpy()

for p in P_VALUES:
    xj_A = boundary_results[("A", p)]["xj"]
    xj_D = boundary_results[("D", p)]["xj"]
    identical = np.array_equal(xj_A, xj_D)
    diff_count = (xj_A != xj_D).sum()
    r_A, _ = pearsonr(xj_A, income_arr)
    r_D, _ = pearsonr(xj_D, income_arr)
    m2_A = boundary_results[("A", p)]["M2_coverage_800m"]
    print(f"p={p}: xj identical A vs D: {identical} ({diff_count} LSOAs differ) | "
          f"r(xj,income) A={r_A:.4f} D={r_D:.4f} | M2={m2_A:.1%}")
